<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

sys.path.append(os.path.abspath(".."))
from src.utils import normalize_images, set_seed
from src.metrics import classification_metrics
from src.experiment import setup_experiment, start_run, log_params, log_metrics, measure_training_time


In [3]:
setup_experiment("assignment")

2026/05/24 20:18:57 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/24 20:18:57 INFO mlflow.store.db.utils: Updating database tables
2026/05/24 20:18:58 INFO mlflow.tracking.fluent: Experiment with name 'assignment' does not exist. Creating a new experiment.


# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
import os


def load_data(seed=42, subset_size=None, cache_path='cifar10_cached.npz'):
    # Try to load from cache to avoid re-downloading
    if os.path.exists(cache_path):
        data = np.load(cache_path)
        X_full = data['X']
        y_full = data['y']
    else:
        try:
            from tensorflow.keras.datasets import cifar10
            (X_full, y_full), _ = cifar10.load_data()
        except Exception:
            import urllib.request, pickle, tarfile
            folder = "cifar-10-batches-py"
            if not os.path.exists(folder):
                url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
                fname = "cifar-10-python.tar.gz"
                urllib.request.urlretrieve(url, fname)
                with tarfile.open(fname, "r:gz") as t:
                    t.extractall()
            def read_batch(f):
                with open(f, "rb") as fo:
                    return pickle.load(fo, encoding="bytes")
            batches = [read_batch(f"{folder}/data_batch_{i}") for i in range(1, 6)]
            X_full = np.concatenate([b[b"data"] for b in batches])
            y_full = np.concatenate([b[b"labels"] for b in batches])
        # Save cache for future runs (save raw image arrays, not flattened)
        try:
            np.savez(cache_path, X=X_full, y=y_full)
        except Exception:
            pass

    X = X_full.reshape(X_full.shape[0], -1).astype(np.float32)
    X = normalize_images(X)
    y = y_full.ravel().astype(np.int64)

    if subset_size is not None:
        X = X[:subset_size]
        y = y[:subset_size]

    set_seed(seed)
    return train_test_split(X, y, test_size=0.2, random_state=seed)


# Fast default for interactive runs: use a smaller subset to avoid long downloads
X_train, X_val, y_train, y_val = load_data(seed=42, subset_size=5000)

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")

KeyboardInterrupt: 

**Respostas — Questão 1**

1. **Formato original das imagens:** (50000, 32, 32, 3) — 50.000 imagens com altura 32, largura 32 e 3 canais RGB.

2. **Features após o flatten:** 32 x 32 x 3 = **3.072 features** por imagem.

3. **Por que o flatten é necessário para uma MLP:** A MLP opera sobre vetores 1D. Ela não processa tensores multidimensionais diretamente — cada neurônio da camada de entrada corresponde a uma feature escalar.

4. **Importância da normalização:** Mantém os valores em [0, 1], colocando todas as features na mesma escala. Isso estabiliza os gradientes, acelera a convergência e evita que pixels com valores altos dominem o aprendizado.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
from sklearn.neural_network import MLPClassifier


def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed=42, max_iter=100, batch_size=256):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        solver="adam",
        learning_rate_init=learning_rate,
        max_iter=max_iter,
        batch_size=batch_size,
        random_state=seed,
        early_stopping=True,
        n_iter_no_change=10,
        tol=1e-4,
        verbose=False,
    )
    model.fit(X_train, y_train)
    return model

**Respostas — Questão 2**

1. **Parâmetros na primeira camada:** Com hidden_layers=(128,) e entrada de 3.072 features: 3.072 x 128 pesos + 128 bias = **393.344 parâmetros**.

2. **Função da ativação ReLU:** ReLU(x) = max(0, x). Introduz não-linearidade sem saturar para valores positivos, reduzindo o problema de vanishing gradient e acelerando o treinamento.

3. **Por que MLPs têm muitos parâmetros com imagens:** A MLP é fully connected — cada neurônio recebe todas as 3.072 features como entrada. Com uma camada de 128 neurônios isso já gera ~393 mil parâmetros, e o número cresce proporcionalmente ao produto entre features e neurônios.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
from src.metrics import classification_metrics


def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return classification_metrics(y_test, y_pred)


# Demonstração
model_demo = train_mlp(X_train, y_train, activation="relu", hidden_layers=(64,), learning_rate=0.001, seed=42)
metrics_demo = evaluate(model_demo, X_val, y_val)
print(pd.DataFrame([metrics_demo]).to_string(index=False))


**Respostas — Questão 3**

1. **O que a accuracy representa:** A proporção de predições corretas sobre o total de exemplos. No caso atual, o modelo obteve accuracy de 0.335, ou seja, acertou cerca de 33,5% das imagens.

2. **Diferença entre precision e recall:**
   - *Precision* = TP / (TP + FP): dos exemplos classificados como uma classe, quantos realmente pertencem a ela.
   - *Recall* = TP / (TP + FN): dos exemplos que realmente pertencem a uma classe, quantos o modelo identificou corretamente.

3. **Quando o F1-score é importante:** Quando as classes são desbalanceadas ou quando tanto falsos positivos quanto falsos negativos têm custo relevante. O F1-score é a média harmônica de precision e recall, penalizando modelos que equilibram mal essas duas medidas.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
configs = [
    {"activation": "relu", "hidden_layers": (128, 64), "learning_rate": 0.001, "max_iter": 100, "batch_size": 200},
    {"activation": "relu", "hidden_layers": (64,),     "learning_rate": 0.01,  "max_iter": 100, "batch_size": 200},
    {"activation": "tanh", "hidden_layers": (128, 64), "learning_rate": 0.001, "max_iter": 100, "batch_size": 200},
]

for cfg in configs:
    with start_run():
        log_params({
            "activation":    cfg["activation"],
            "hidden_layers": str(cfg["hidden_layers"]),
            "learning_rate": cfg["learning_rate"],
            "max_iter":      cfg["max_iter"],
            "batch_size":    cfg["batch_size"],
        })

        model, training_time = measure_training_time(
            train_mlp,
            X_train,
            y_train,
            activation=cfg["activation"],
            hidden_layers=cfg["hidden_layers"],
            learning_rate=cfg["learning_rate"],
            seed=42,
            max_iter=cfg["max_iter"],
            batch_size=cfg["batch_size"],
        )

        metrics = evaluate(model, X_val, y_val)
        metrics["training_time"] = training_time

        log_metrics(metrics)

        label = f"act={cfg['activation']} layers={cfg['hidden_layers']} lr={cfg['learning_rate']}"
        print(label)

        print(pd.DataFrame([metrics]).to_string(index=False))
        print()

**Respostas — Questão 4**

1. **Experimento com melhor desempenho:** O experimento com activation='tanh', hidden_layers=(128, 64) e learning_rate=0.001 obteve a melhor performance, com accuracy 0.387 e f1_score 0.38279.

2. **Configuração com maior estabilidade:** A configuração tanh + (128, 64) + 0.001 foi a mais estável entre as testadas. A configuração menor relu + (64,) + 0.01 teve desempenho muito ruim e não se apresentou como estável.

3. **Benefício do rastreamento experimental:** Permite comparar diferentes configurações de hiperparâmetros de forma sistemática e reproduzível, identificar quais combinações produzem os melhores resultados e evitar conclusões baseadas em execuções isoladas.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
activations   = ["logistic", "tanh", "relu"]
hidden_layers = (128, 64)
learning_rate = 0.001
seed          = 42

results_q5 = []

for activation in activations:
    with start_run(run_name=f"q5_act_{activation}"):
        log_params({
            "activation":    activation,
            "hidden_layers": str(hidden_layers),
            "learning_rate": learning_rate,
        })

        model, training_time = measure_training_time(
            train_mlp,
            X_train,
            y_train,
            activation=activation,
            hidden_layers=hidden_layers,
            learning_rate=learning_rate,
            seed=seed,
        )

        metrics = evaluate(model, X_val, y_val)
        metrics["training_time"] = training_time

        log_metrics(metrics)

        results_q5.append({"activation": activation, **metrics})

print(pd.DataFrame(results_q5).set_index("activation").to_string())


**Respostas — Questão 5**

1. **Melhor convergência:** Logistic apresentou o melhor desempenho final nos resultados registrados, com accuracy 0.395, precision 0.401941 e f1_score 0.393671.

2. **Maior estabilidade:** ReLU foi o meio termo mais equilibrado entre accuracy e f1, enquanto tanh ficou abaixo em todas as métricas neste experimento.

3. **Diferenças significativas:** Sim. Logistic teve o desempenho mais alto, ReLU ficou em segundo lugar, e tanh apresentou os menores valores de accuracy e f1.

4. **Por que ReLU é amplamente utilizada em Deep Learning:** ReLU não satura para valores positivos, evitando o vanishing gradient. É computacionalmente simples (max(0, x)), acelera o treinamento e funciona bem em redes profundas.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
architectures = [(32,), (64,), (128, 64), (256, 128)]
activation    = "relu"
learning_rate = 0.001
seed          = 42

results_q6 = []

for arch in architectures:
    with start_run(run_name=f"q6_arch_{arch}"):
        log_params({
            "activation":    activation,
            "hidden_layers": str(arch),
            "learning_rate": learning_rate,
        })

        model, training_time = measure_training_time(
            train_mlp,
            X_train,
            y_train,
            activation=activation,
            hidden_layers=arch,
            learning_rate=learning_rate,
            seed=seed,
        )

        metrics = evaluate(model, X_val, y_val)
        metrics["training_time"] = training_time

        log_metrics(metrics)

        results_q6.append({"architecture": str(arch), **metrics})

print(pd.DataFrame(results_q6).set_index("architecture").to_string())


**Respostas — Questão 6**

1. **Redes maiores sempre melhoram os resultados?** Não necessariamente. Redes maiores têm maior capacidade, mas nem sempre trazem melhor accuracy; neste caso, a arquitetura maior (256, 128) não superou a arquitetura (128, 64).

2. **Melhor tradeoff:** A arquitetura (128, 64) apresentou o melhor equilíbrio entre desempenho e custo, com accuracy 0.359 e f1_score 0.364185.

3. **Sinais de overfitting:** Não houve um overfitting claro, mas os modelos maiores demonstraram retorno decrescente. A arquitetura (256, 128) aumentou o custo de treinamento sem melhorar a accuracy.

4. **Maior custo computacional:** A arquitetura (256, 128) teve o maior custo de treinamento, com tempo de 19.847757 segundos no experimento.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
learning_rates = [0.1, 0.01, 0.001]
hidden_layers  = (128, 64)
activation     = "relu"
seed           = 42

results_q7 = []

for lr in learning_rates:
    with start_run(run_name=f"q7_lr_{lr}"):
        log_params({
            "activation":    activation,
            "hidden_layers": str(hidden_layers),
            "learning_rate": lr,
        })

        model, training_time = measure_training_time(
            train_mlp,
            X_train,
            y_train,
            activation=activation,
            hidden_layers=hidden_layers,
            learning_rate=lr,
            seed=seed,
        )

        metrics = evaluate(model, X_val, y_val)
        metrics["training_time"] = training_time

        log_metrics(metrics)

        results_q7.append({"learning_rate": lr, **metrics})

print(pd.DataFrame(results_q7).set_index("learning_rate").to_string())


**Respostas — Questão 7**

1. **Melhor desempenho:** O melhor desempenho foi com learning rate 0.001, que obteve accuracy 0.359 e f1_score 0.364185.

2. **Maior instabilidade:** Learning rate 0.1 foi o mais instável, com accuracy 0.092 e f1_score 0.015502, indicando que o treinamento não se ajustou bem.

3. **O que acontece com learning rate muito alto:** O modelo pode oscilar ou divergir, como aconteceu com 0.1, resultando em performance muito ruim.

4. **O que acontece com learning rate muito baixo:** A convergência é lenta e pode ser insuficiente dentro do número de iterações disponíveis, mas tende a ser mais estável; em nossos resultados, 0.001 foi o melhor compromisso.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

## Discussão Final — Questão 8

### Comportamento da loss
Em todas as configurações, a loss de treino diminuiu progressivamente ao longo das iterações. Com learning rate alto (0.1) houve oscilação visível; com learning rate baixo (0.001) a queda foi suave e estável.

### Impacto do learning rate
O learning rate é o hiperparâmetro mais sensível: valores muito altos causam oscilação ou divergência; valores muito baixos tornam o treinamento lento e podem resultar em underfitting dentro do número de épocas disponíveis. O valor 0.001 se mostrou o melhor compromisso nos experimentos atuais.

### Impacto da arquitetura
Arquiteturas maiores possuem maior capacidade de representação, mas nem sempre melhoram a accuracy. A arquitetura (128, 64) apresentou o melhor tradeoff entre desempenho e custo, enquanto a arquitetura (256, 128) aumentou o custo sem ganhos claros.

### Impacto das funções de ativação
Neste conjunto de experimentos, logistic apresentou o melhor desempenho final, seguido por ReLU; tanh ficou abaixo dos demais em accuracy e f1. Isso mostra que a escolha da ativação pode alterar significativamente o comportamento da MLP.

### Comportamento do treinamento
Os modelos com learning rate 0.001 foram os mais estáveis e precisos. O learning rate 0.1 apresentou desempenho ruim e indicou que o passo de atualização era grande demais para esta rede e conjunto de dados.

### Limitações da MLP para imagens
A MLP trata cada pixel de forma independente após o flatten, destruindo a estrutura espacial da imagem. Pixels vizinhos perdem sua relação de vizinhança, e a rede não é invariante a translações. Além disso, o número de parâmetros cresce de forma proporcional ao produto entre features de entrada e neurônios, resultando em modelos muito grandes para imagens de resolução moderada.

### Relação entre backpropagation e aprendizado
O backpropagation calcula o gradiente da função de loss em relação a cada peso da rede, propagando o erro da camada de saída para as camadas anteriores via regra da cadeia. Esse gradiente é utilizado pelo otimizador para atualizar os pesos em direção ao mínimo da loss, permitindo que cada camada aprenda representações progressivamente mais abstratas dos dados.

---

### Respostas

1. **Melhor configuração final:** Nos experimentos realizados, a melhor configuração registrada foi logistic + arquitetura (128, 64) + learning rate 0.001, com accuracy 0.395 e f1_score 0.3937.

2. **Principais dificuldades:** Alto custo computacional (3.072 features × muitos parâmetros), sensibilidade ao learning rate e escolha da função de ativação, além da limitação da MLP em explorar a estrutura espacial das imagens.

3. **Por que MLPs têm limitações para imagens:** O flatten destrói a informação espacial — a rede não aproveita que pixels vizinhos são correlacionados. CNNs, por outro lado, exploram localidade e invariância a translação, sendo muito mais eficientes para visão computacional.

4. **Como o backpropagation contribui:** Ele torna viável o treinamento de redes com múltiplas camadas ao distribuir o sinal de erro eficientemente por toda a rede. Sem backpropagation, não haveria como calcular os gradientes das camadas intermediárias de forma escalável.